# Malinois MPRA Activity Scoring

![Malinois MPRA Activity Scoring](https://proto-bio.github.io/proto-assets/images/tool/malinois/hero.png)

Malinois is the regulatory-sequence convolutional neural network at the core of CODA (Computational Optimization of DNA Activity), published by Gosai et al. (2024). It predicts massively parallel reporter assay (MPRA) activity for 200 bp DNA inserts in three cell contexts — K562, HepG2, and SK-N-SH — and was trained on aggregated log2 fold-change measurements to enable machine-guided design of cell-type-targeting cis-regulatory elements.

This notebook demonstrates both Malinois tools: `run_malinois_score`, which returns reverse-complement-averaged raw activity for candidate inserts, and `run_malinois_gradient`, which reports each requested cell type's raw activity and its gradient with respect to relaxed `B x L x 4` DNA logits for Fast SeqProp-style sequence design.

In [1]:
from proto_tools.utils.notebook_docs import (
    display_overview,
    display_api_reference,
    display_docs_section,
    display_doc_link,
    display_available_tools,
)

display_doc_link("malinois")
display_overview("malinois")
display_docs_section("malinois", "Background")

# Malinois

Malinois is the CODA/BODA2 convolutional neural network for predicting MPRA-measured regulatory DNA activity from 200 bp inserts. This toolkit scores sequences in K562, HepG2, and SK-N-SH cell contexts and exposes a differentiable activity-gradient call for sequence design.

Malinois is the regulatory sequence model used in CODA ([Gosai et al., 2024](https://doi.org/10.1038/s41586-024-08070-z)) for machine-guided design of cell-type-targeting cis-regulatory elements. The model adapts the Basset-style convolutional architecture to MPRA data and predicts activity from a fixed 200 nucleotide insert after adding the assay flanks expected by the published checkpoint.

The model returns one raw activity value for each supported cell context: K562, HepG2, and SK-N-SH. The scoring wrapper averages forward and reverse-complement predictions and returns selected raw outputs. The gradient wrapper reports those raw scores and their derivatives with respect to relaxed A,C,G,T logits, one cell type at a time, matching the Fast SeqProp-style design path used for regulatory DNA optimization.

## Available tools

In [2]:
display_available_tools("malinois")

- **`run_malinois_gradient()`** — Differentiate Malinois MPRA cell-type activity for relaxed DNA logits
- **`run_malinois_score()`** — Score regulatory DNA activity using the Malinois MPRA model

### `run_malinois_score`

Given one or more 200 bp DNA inserts, this function adds the MPRA assay flanks expected by the published checkpoint, runs the forward and reverse-complement sequences through Malinois, and returns the averaged raw activity for each requested cell type. It is the tool to use when ranking or screening regulatory DNA candidates by predicted K562, HepG2, or SK-N-SH activity. Scoring is deterministic and batched, so `batch_size` changes throughput without changing results.

In [3]:
from proto_tools.tools.sequence_scoring.malinois import (
    MalinoisScoreConfig,
    MalinoisScoreInput,
    run_malinois_score,
)

In [4]:
# Display input docs
display_api_reference("malinois", "input", "run_malinois_score")

# Two real 200 bp MPRA inserts from Gosai et al. 2024 (Table S2): a strongly
# active enhancer (high measured activity in all three cell types) and a
# near-silent element (measured activity near zero).
active_enhancer = (
    "CCCACCGGGAGCCTCGGAAGGACCGGCCTCCCCTCACTCTAAGCATGCGCGTTAAAGCGAC"
    "AGCATAGGCTATAGAAAGCTTTATGAGAATAGCTGGGAGCCATACAGCCGCAGGGCCGCGT"
    "GGCCTAATGGATAAGGCGTCTGATTCCGGATCAGAAGATTGAGGGTTCGAGTCCCTTCGTG"
    "GTCGTTGCCATGTTAAC"
)
inactive_element = (
    "CATAAACAATTGCTATTCACAAATTATTCGAATCAAATCAGGTCTCAGCAAATTCCCAACT"
    "CTGCAATGTCATTGCAATGTTATATATAAAAGAGATCCTGAAATAAGTAAGCTATTATCAT"
    "ATATTTGAAAATTCGGCCAGGCATGATAGCATGCGCCTGTTATCCCAGCTACTCAGGCTGG"
    "GAAGGCTGAAGTGAGAG"
)
inputs = MalinoisScoreInput(sequences=[active_enhancer, inactive_element])

**Input** — `MalinoisScoreInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>sequences</code> | <code>list[str]</code> | required | DNA insert sequence(s) to score with Malinois |

In [5]:
# Display config docs
display_api_reference("malinois", "config", "run_malinois_score")

config = MalinoisScoreConfig(
    cell_types=["K562", "HepG2", "SKNSH"],
    batch_size=2,
    device="cuda",  # Change to "cpu" if no GPU available
)

**Config** — `MalinoisScoreConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>device</code> | <code>str</code> | <code>'cuda'</code> | Device to run Malinois inference on |
| <code>cell_types</code> | <code>list[Literal['K562', 'HepG2', 'SKNSH']]</code> | <code>['K562', 'HepG2', 'SKNSH']</code> | Malinois cell-type outputs to return. |
| <code>seq_length</code> | <code>int</code> | <code>200</code> | Expected DNA insert length before MPRA flank padding. |
| <code>artifact_path</code> | <code>str</code> | <code>''</code> | Optional local artifact tarball path; empty uses the managed cache download. |
| <code>malinois_dir</code> | <code>str</code> | <code>''</code> | Optional local Malinois metadata directory; empty uses the managed cache extraction. |
| <code>batch_size</code> | <code>int</code> | <code>8</code> | Sequences per GPU forward pass; raise for throughput, lower if OOM |

In [6]:
result = run_malinois_score(inputs, config)

# Malinois recovers the measured contrast: high predicted activity for the
# active enhancer, near-zero for the inactive element.
labels = ["active_enhancer", "inactive_element"]
[{"insert": labels[idx], **item.scores} for idx, item in enumerate(result.results)]

Running run_malinois_score [00:00]

[{'insert': 'active_enhancer',
  'K562': 6.42901611328125,
  'HepG2': 4.607794761657715,
  'SKNSH': 5.777548789978027},
 {'insert': 'inactive_element',
  'K562': 0.40050989389419556,
  'HepG2': 0.1743352711200714,
  'SKNSH': 0.15775281190872192}]

### `run_malinois_gradient`

For gradient-based DNA design, this function takes batched relaxed logits of shape `(B, L, 4)` in `A,C,G,T` order, relaxes them to nucleotide probabilities at the given softmax temperature, and reports one term per requested cell type, each carrying that cell's raw predicted activity. A single call therefore supplies both sides of an on-target / off-target trade-off, and the caller composes a goal per term. With `compute_gradient=True` each term also carries that activity's gradient, with the same shape as the input logits, ready to drive an optimizer loop.

In [7]:
import numpy as np

from proto_tools.tools.sequence_scoring.malinois import (
    MalinoisActivityTerm,
    MalinoisGradientConfig,
    MalinoisGradientInput,
    run_malinois_gradient,
)

In [8]:
# Display input docs
display_api_reference("malinois", "input", "run_malinois_gradient")

# A batch of two random relaxed 200 bp candidates (B, L, 4) in A,C,G,T order
rng = np.random.default_rng(0)
logits = rng.normal(size=(2, 200, 4)).tolist()
gradient_inputs = MalinoisGradientInput(inputs=logits, temperature=1.0)

**Input** — `MalinoisGradientInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>temperature</code> | <code>float &#124; None</code> | <code>1.0</code> | Softmax temperature for softmax(logits / T); None consumes probabilities directly. |
| <code>inputs</code> | <code>list[NucleotideSequenceLogits]</code> | required | One relaxed (L, 4) DNA state per candidate in A,C,G,T order |

In [9]:
# Display config docs
display_api_reference("malinois", "config", "run_malinois_gradient")

# Differentiate the on-target K562 activity and the off-target HepG2 activity
gradient_config = MalinoisGradientConfig(
    gradient_terms=[
        MalinoisActivityTerm(cell_type="K562"),
        MalinoisActivityTerm(cell_type="HepG2"),
    ],
    compute_gradient=True,
    device="cuda",  # Change to "cpu" if no GPU available
)

**Config** — `MalinoisGradientConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>device</code> | <code>str</code> | <code>'cuda'</code> | Device to run Malinois inference on |
| <code>soft</code> | <code>float</code> | <code>1.0</code> | Blend raw input (0) toward softmax probabilities (1); 1.0 recommended for almost all uses. |
| <code>hard</code> | <code>float</code> | <code>0.0</code> | Straight-through coefficient; 1 forwards a hard one-hot to the model. |
| <code>gradient_terms</code> | <code>list[MalinoisActivityTerm]</code> | <code>[MalinoisActivityTerm(id='activity', cell_type='K562')]</code> | Malinois cell types to differentiate, by name or as instances. |
| <code>seq_length</code> | <code>int</code> | <code>200</code> | Expected DNA insert length before MPRA flank padding. |
| <code>artifact_path</code> | <code>str</code> | <code>''</code> | Optional local artifact tarball path; empty uses the managed cache download. |
| <code>malinois_dir</code> | <code>str</code> | <code>''</code> | Optional local Malinois metadata directory; empty uses the managed cache extraction. |
| <code>compute_gradient</code> | <code>bool</code> | <code>True</code> | Run backward pass and return gradient; set False for forward-only scoring. |

In [10]:
gradient_result = run_malinois_gradient(gradient_inputs, gradient_config)
# One result per candidate, each with one raw activity and gradient per requested cell
{
    "raw_scores": [
        {term.term.cell_type: round(term.term_value, 3) for term in r.terms}
        for r in gradient_result.results
    ],
    "gradient_shape": np.array(gradient_result.results[0].terms[0].gradient).shape,
}

Running run_malinois_gradient [00:00]

{'raw_scores': [{'K562': 0.588, 'HepG2': 0.587},
  {'K562': 0.578, 'HepG2': 0.574}],
 'gradient_shape': (200, 4)}